# Visual-Doc Assistant — Phase 2
## Vision Model Integration & Embedding Generation

**Objective:** Integrate ColPali (Vision-Language Model) as the **intelligence core** — generate visual embeddings from all 26 page images (Phase 1 output), store them in ChromaDB, and verify that both image and query embeddings share the same semantic space (proving retrieval-readiness for Phase 3).

---
### Phase Context
```
Phase 1  →  PDF → Images + ChromaDB initialised (0 embeddings)
Phase 2  →  Images → ColPali → 128-dim Embeddings → ChromaDB       THIS NOTEBOOK
           +Query → ColPali → 128-dim Query Vector → Sanity Check  THIS NOTEBOOK
Phase 3  →  Query Retrieval → Top-K Pages → Gemini → Answer
```

### Why ColPali?
Traditional RAG uses OCR to extract text — losing diagrams, tables, and layout.  
ColPali (ICLR 2025) feeds each **page image** directly into PaliGemma-3B (SigLIP vision + Gemma-2B language model) and produces **128-dim patch-level embeddings** that capture both visual and semantic content simultaneously.  
The same model encodes text queries into the **same 128-dim space** — enabling direct image-to-query similarity matching without any OCR.

> **GPU Required:** Go to `Runtime → Change runtime type → T4 GPU` before running.

## Step 1 — Mount Drive & Install Dependencies

All packages are installed into `/content/drive/MyDrive/Visual-Doc/deps` on your Google Drive instead of Colab's ephemeral environment.  
**Why:** Colab resets its local disk on every session. Installing to Drive means you only download the packages once — future sessions just mount Drive and load from there instantly.

```
Visual-Doc/
├── processed_images/     ← Phase 1 output (26 PNGs)
├── chroma_db_storage/    ← Phase 1 output (ChromaDB)
└── deps/                 ← Phase 2: all Python packages stored here
```

In [ ]:
# Mount Google Drive — Phase 1 images and ChromaDB are stored here
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install peft==0.14.0 torchao==0.16.0 colpali-engine==0.3.7 -q --no-warn-conflicts

In [ ]:
import sys, os, subprocess

WHEELS_DIR = './deps'
os.makedirs(WHEELS_DIR, exist_ok=True)

# Ensure compatible versions are pre-installed in Colab's env
!pip install peft==0.14.0 torchao==0.16.0 -q --no-warn-conflicts

def pip_run(cmd, label):
    """Run a pip command, print a clean one-line result."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    our = ['colpali', 'chromadb', 'pillow', 'tqdm']
    errs = [l for l in result.stderr.splitlines()
            if 'ERROR' in l and any(p in l.lower() for p in our)]
    if errs:
        print(f'  {label}:')
        for e in errs: print(f'   {e}')
    else:
        print(f' {label}')

CORE = ['colpali-engine==0.3.7', 'chromadb', 'Pillow<12.0.0', 'tqdm']

# ── Check if wheels already cached on Drive ────────────────────────────────────
cached_wheels = [f for f in os.listdir(WHEELS_DIR) if f.endswith('.whl')]

if cached_wheels:
    print(f' Found {len(cached_wheels)} cached wheels on Drive — installing from cache...')
    pip_run(
        [sys.executable, '-m', 'pip', 'install',
         '--no-index', '--find-links', WHEELS_DIR,
         '--quiet', '--no-warn-conflicts'] + CORE,
        'Core packages (from Drive cache)'
    )
else:
    print(' No cache — downloading and saving wheels to Drive...')
    print('   (Only happens once — future sessions use the Drive cache)')
    pip_run(
        [sys.executable, '-m', 'pip', 'download',
         '--dest', WHEELS_DIR, '--quiet'] + CORE,
        f'Wheels downloaded to {WHEELS_DIR}'
    )
    pip_run(
        [sys.executable, '-m', 'pip', 'install',
         '--no-index', '--find-links', WHEELS_DIR,
         '--quiet', '--no-warn-conflicts'] + CORE,
        'Core packages installed'
    )

# ── Verify imports ─────────────────────────────────────────────────────────────
print()
try:
    import colpali_engine, chromadb
    print(f' colpali_engine : {colpali_engine.__file__}')
    print(f' chromadb       : {chromadb.__version__}')
    print(f'\n Wheel cache    : {WHEELS_DIR}')
    print(f'   Cached files   : {len(os.listdir(WHEELS_DIR))}')
    print('\n   Next session: mount Drive → run this cell → done in ~30 sec.')
except ImportError as e:
    print(f' Import failed: {e}')

## Step 2 — Load ColPali Vision-Language Model

ColPali is PaliGemma-3B fine-tuned with contrastive loss and a ColBERT projection layer.  
It projects every image patch (and text token) into a shared **128-dimensional embedding space**.  
Loading in `bfloat16` halves VRAM usage with negligible quality loss — fits on T4 GPU (16 GB).

In [ ]:
import torch
from PIL import Image
from IPython.display import display, Image as IPImage
from colpali_engine.models import ColPali, ColPaliProcessor

# ── Configuration ──────────────────────────────────────────────────────────────
MODEL_NAME = 'vidore/colpali-v1.2'
EMBED_DIM  = 128
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device   : {DEVICE}')
print(f'Embed dim: {EMBED_DIM}')
if DEVICE == 'cpu':
    print('  No GPU detected — switch to T4 GPU runtime before continuing.')

# ── Load model & processor ─────────────────────────────────────────────────────
print(f'\nLoading {MODEL_NAME} ...')
print('(First run downloads ~6 GB from HuggingFace — subsequent runs use the Colab cache)')

model = ColPali.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,   # Half-precision to save VRAM
).eval().to(DEVICE)               # Explicitly move to GPU instead of device_map='auto'

processor = ColPaliProcessor.from_pretrained(MODEL_NAME)

print(f'\n ColPali loaded on {DEVICE}')
print(f'   Backbone : PaliGemma-3B (SigLIP-So400m vision + Gemma-2B language model)')
print(f'   Projection: each patch/token → {EMBED_DIM}-dim vector (ColBERT style)')

## Step 3 — Verify Phase 1 Outputs

Confirm the 26 page images and the empty ChromaDB collection from Phase 1 are accessible.

In [ ]:
import os
import numpy as np
import chromadb

# ── Phase 1 output paths ───────────────────────────────────────────────────────
IMAGE_FOLDER = './processed_images'
DB_PATH      = './chroma_db_storage'

# ── Verify images ──────────────────────────────────────────────────────────────
image_files = sorted(
    [f for f in os.listdir(IMAGE_FOLDER) if f.endswith('.png')],
    key=lambda x: int(x.split('_')[1].split('.')[0])
)
print(f'Images found : {len(image_files)}')
print(f'   First: {image_files[0]}  |  Last: {image_files[-1]}')

# Quick sanity — open first and last
for fname in [image_files[0], image_files[-1]]:
    img = Image.open(os.path.join(IMAGE_FOLDER, fname))
    print(f'   {fname}: size={img.size}, mode={img.mode}')

# ── Verify ChromaDB ────────────────────────────────────────────────────────────
client     = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_or_create_collection(
    name='visual_doc_collection',
    metadata={'hnsw:space': 'cosine'}   # Cosine similarity — same as Phase 1
)
print(f'\n ChromaDB collection : "{collection.name}"')
print(f'   Embeddings stored   : {collection.count()}')   # 0 from Phase 1
print('\n Phase 1 outputs verified. Ready to embed.')

## Step 4 — Define Embedding Functions

ColPali uses the **same model and the same 128-dim space** for both images and text queries.  
This shared space is the key architectural insight — it means a text query can be directly compared to a page image embedding using cosine similarity, with no OCR or text extraction needed.

```
Page Image  →  SigLIP patches  →  Gemma-2B  →  Projection  →  (N_patches × 128)
                                                                    ↓ mean-pool
                                                               (128-dim) → ChromaDB

Text Query  →  Gemma-2B tokenizer  →  Projection  →  (N_tokens × 128)
                                                           ↓ mean-pool
                                                       (128-dim) → cosine search
```

> **On mean-pooling:** The paper uses full late-interaction (MaxSim across all patch vectors). ChromaDB stores single vectors, so we mean-pool as a practical trade-off. Full late-interaction via PyColBERT is a Phase 3 upgrade path.

In [ ]:
@torch.no_grad()
def embed_page_image(image: Image.Image) -> np.ndarray:
    """
    Generate a 128-dim embedding for a single page image using ColPali.

    Pipeline:
      1. ColPaliProcessor converts PIL image → model input tensors
      2. ColPali (PaliGemma-3B) encodes image → patch embeddings shape (1, N_patches, 128)
      3. Mean-pool over patches → shape (128,)
      4. Return as float32 numpy array

    Args:
        image: PIL Image of a document page (300 DPI PNG from Phase 1)
    Returns:
        numpy array of shape (128,), dtype float32
    """
    inputs          = processor.process_images([image]).to(DEVICE)
    patch_embeddings = model(**inputs)                         # (1, N_patches, 128)
    pooled          = patch_embeddings.mean(dim=1).squeeze(0)  # (128,)
    return pooled.cpu().float().numpy()


@torch.no_grad()
def embed_query(query_text: str) -> np.ndarray:
    """
    Generate a 128-dim embedding for a text query using ColPali's text encoder.
    Uses the SAME model and projection layer as embed_page_image —
    query and image embeddings live in the same semantic space.

    Note: ColPaliProcessor automatically appends 5 <unused0> tokens to the query
    (ColBERT query augmentation — acts as soft query expansion).

    Args:
        query_text: natural language question string
    Returns:
        numpy array of shape (128,), dtype float32
    """
    inputs          = processor.process_queries([query_text]).to(DEVICE)
    token_embeddings = model(**inputs)                          # (1, N_tokens, 128)
    pooled          = token_embeddings.mean(dim=1).squeeze(0)  # (128,)
    return pooled.cpu().float().numpy()


# ── Sanity test both functions ─────────────────────────────────────────────────
test_image     = Image.open(os.path.join(IMAGE_FOLDER, image_files[0])).convert('RGB')
test_img_emb   = embed_page_image(test_image)
test_query_emb = embed_query('What is the system architecture?')

print('Image embedding  — shape:', test_img_emb.shape,   '| dtype:', test_img_emb.dtype)
print('Query embedding  — shape:', test_query_emb.shape,  '| dtype:', test_query_emb.dtype)

# Compute cosine similarity between query and page 1 as proof of shared space
cos_sim = np.dot(test_img_emb, test_query_emb) / (
    np.linalg.norm(test_img_emb) * np.linalg.norm(test_query_emb)
)
print(f'\nCosine similarity (query ↔ page 1): {cos_sim:.4f}')
print('(Non-zero value confirms both vectors share the same embedding space)')
print('\n Both embedding functions working correctly.')

## Step 5 — Batch Embed All 26 Pages & Store in ChromaDB

Process all page images in batches of 4 (safe for T4 16 GB VRAM).  
Each page is stored with its 128-dim embedding and metadata.  
The loop is **resume-safe** — skips pages already embedded if interrupted.

In [ ]:
from tqdm import tqdm
import time

BATCH_SIZE = 4   # Reduce to 2 if CUDA OOM errors occur

print(f'Embedding {len(image_files)} pages in batches of {BATCH_SIZE}...')
print(f'Estimated time: ~{len(image_files)*3//60}–{len(image_files)*5//60} minutes on T4 GPU\n')

total_added   = 0

total_skipped = 0
start_time    = time.time()

for batch_start in tqdm(range(0, len(image_files), BATCH_SIZE), desc='Embedding pages'):
    batch_files = image_files[batch_start : batch_start + BATCH_SIZE]

    ids, embeddings, metadatas, documents = [], [], [], []

    for filename in batch_files:
        page_num = int(filename.split('_')[1].split('.')[0])
        doc_id   = f'page_{page_num}'

        # Resume-safety: skip if already embedded in a previous run
        if collection.get(ids=[doc_id])['ids']:
            total_skipped += 1
            continue

        image_path = os.path.join(IMAGE_FOLDER, filename)
        image      = Image.open(image_path).convert('RGB')   # Ensure RGB (some PNGs are RGBA)
        embedding  = embed_page_image(image)

        ids.append(doc_id)
        embeddings.append(embedding.tolist())
        metadatas.append({
            'page_number': page_num,
            'image_path' : image_path,
            'filename'   : filename,
            'doc_name'   : '2407.01449v6',   # Source PDF — useful when indexing multiple docs
        })
        documents.append(f'Page {page_num}')   # ChromaDB requires a text document field

    if ids:
        collection.add(
            ids        = ids,
            embeddings = embeddings,
            metadatas  = metadatas,
            documents  = documents,
        )
        total_added += len(ids)

elapsed = time.time() - start_time
print(f'\n Embedding complete!')
print(f'   Pages added  : {total_added}')
print(f'   Pages skipped: {total_skipped} (already in DB)')
print(f'   Total in DB  : {collection.count()}')
print(f'   Time elapsed : {elapsed/60:.1f} minutes')

## Step 6 — Verify Stored Embeddings

Pull back sample entries to confirm embedding shape, metadata, and DB health.

In [ ]:
print('=' * 55)
print('       ChromaDB Verification Report')
print('=' * 55)

total = collection.count()
print(f'\nCollection      : {collection.name}')
print(f'Total embeddings: {total}')
print(f'Expected        : {len(image_files)}')
print(f'Status          : {" All pages embedded" if total == len(image_files) else "⚠️  Mismatch — re-run Step 5"}')

# Sample 3 entries across the document
sample = collection.get(
    ids     = ['page_1', 'page_13', 'page_26'],
    include = ['embeddings', 'metadatas']
)

print('\n── Sample Entries ──')
for i, doc_id in enumerate(sample['ids']):
    emb  = np.array(sample['embeddings'][i])
    meta = sample['metadatas'][i]
    print(f'\n  ID          : {doc_id}')
    print(f'  Page number : {meta["page_number"]}')
    print(f'  Filename    : {meta["filename"]}')
    print(f'  Doc name    : {meta["doc_name"]}')
    print(f'  Emb shape   : {emb.shape}')     # Should be (128,)
    print(f'  Emb norm    : {np.linalg.norm(emb):.4f}')
    print(f'  Emb sample  : {emb[:4].round(4)}')

print('\n' + '=' * 55)

## Step 7 — End-to-End Embedding Sanity Check (Retrieval Preview)

This is a **Phase 2 verification step** — not Phase 3 retrieval.  
We encode a text query using `embed_query()` and run a similarity search to confirm that:
1. The query embedding and image embeddings are in the same space
2. ChromaDB can perform cosine search correctly
3. The returned pages make semantic sense for the query

This proves the embedding generation pipeline is **retrieval-ready** for Phase 3.

In [ ]:
def retrieval_sanity_check(query_text: str, k: int = 3):
    """
    Phase 2 verification: encode a query and find top-k similar pages.
    Displays retrieved page thumbnails alongside similarity scores.
    No answer generation — that is Phase 3.
    """
    print(f'Query: "{query_text}"\n')

    # Encode the query using the same ColPali model
    query_embedding = embed_query(query_text)

    # Cosine similarity search in ChromaDB
    results = collection.query(
        query_embeddings = [query_embedding.tolist()],
        n_results        = k,
        include          = ['metadatas', 'distances'],
    )

    print(f'Top-{k} retrieved pages:')
    for meta, dist in zip(results['metadatas'][0], results['distances'][0]):
        similarity = 1 - dist   # ChromaDB returns cosine distance; convert to similarity
        print(f'  Page {meta["page_number"]:>2} — cosine similarity: {similarity:.4f}')

        # Show thumbnail of the retrieved page
        img = Image.open(meta['image_path']).convert('RGB')
        img.thumbnail((400, 600))   # Resize for display only
        display(img)

    print()


# ── Run two sanity checks with different query types ───────────────────────────
retrieval_sanity_check('What is the system architecture and pipeline diagram?', k=3)
retrieval_sanity_check('What are the results and performance comparisons?', k=3)

## Step 8 — Embedding Similarity Heatmap

Plot a 26×26 cosine similarity matrix across all pages.  
Pages with similar visual content should show higher similarity — visually confirming the embeddings are semantically meaningful (useful for the project report).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

# Retrieve all embeddings in page order
all_ids = [f'page_{i}' for i in range(1, len(image_files) + 1)]
result  = collection.get(ids=all_ids, include=['embeddings'])

embedding_matrix = np.array(result['embeddings'])        # (26, 128)
sim_matrix       = cosine_similarity(embedding_matrix)   # (26, 26)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(len(all_ids)))
ax.set_yticks(range(len(all_ids)))
ax.set_xticklabels([f'P{i}' for i in range(1, len(image_files)+1)], fontsize=7)
ax.set_yticklabels([f'P{i}' for i in range(1, len(image_files)+1)], fontsize=7)
plt.colorbar(im, ax=ax, label='Cosine Similarity')
ax.set_title('ColPali Embedding Similarity Heatmap — All 26 Pages\n'
             '(Brighter = more visually/semantically similar)', fontsize=12)
plt.tight_layout()
plt.savefig('./embedding_similarity_heatmap.png', dpi=150)
plt.show()

# Stats
np.fill_diagonal(sim_matrix, 0)
max_idx = np.unravel_index(sim_matrix.argmax(), sim_matrix.shape)
print(f'Most similar pair : Pages {max_idx[0]+1} & {max_idx[1]+1}  (sim={sim_matrix.max():.4f})')
print(f'Min similarity    : {sim_matrix.min():.4f}')
print(f'Mean similarity   : {sim_matrix.mean():.4f}')
print('\n Heatmap saved to Google Drive.')

---
## Phase 2 Summary

| Component | Detail |
|---|---|
| Vision Model | ColPali v1.2 — PaliGemma-3B (SigLIP-So400m + Gemma-2B) |
| Embedding Dimension | 128 (ColBERT projection layer) |
| Image Encoding | ~1024 patch vectors per page → mean-pooled to 1 × 128 vector |
| Query Encoding | Text tokens → mean-pooled to 1 × 128 vector (same space as images) |
| Pages Embedded | 26 (all pages from Phase 1) |
| Storage | ChromaDB persistent client — cosine similarity HNSW index |
| Metadata per page | page_number, image_path, filename, doc_name |
| Sanity Check | Query → embed → cosine search → top-K pages displayed |

### What belongs to Phase 3 (not included here)
- Gemini API integration
- Multimodal prompt construction
- Answer generation from retrieved pages
- Streamlit user interface
- Full late-interaction retrieval (PyColBERT / Vespa)

---
*Visual-Doc Assistant — NIT Patna, MCA (DS&I), Course: MC460502*